<style>
.note {padding: 12px 16px; border-left: 5px solid #2563eb; background: #eff6ff; margin: 10px 0;}
.warn {padding: 12px 16px; border-left: 5px solid #d97706; background: #fffbeb; margin: 10px 0;}
.fix  {padding: 12px 16px; border-left: 5px solid #dc2626; background: #fef2f2; margin: 10px 0;}
.exam {padding: 12px 16px; border-left: 5px solid #059669; background: #ecfdf5; margin: 10px 0;}
table {font-size: 95%;}
</style>

# 05 — Bayes’ Theorem

### Updating prior beliefs after observing evidence

**Level:** beginner → advanced  
**Style:** short explanations, worked examples, formulas, runnable code, revision material

## What you will be able to do

- distinguish joint, marginal, and conditional probability
- derive and apply Bayes’ theorem
- understand base-rate effects in diagnostic testing
- connect Bayes to Naive Bayes and Bayesian parameter estimation

## Resource coverage

- Transcript lines 2548–2890: independent/dependent events, marble example, Bayes derivation, house-feature/price application
- Handwritten Bayes resource: page 1 shows dice, coin, marble conditional probability, and theorem; page 2 maps $x_1,x_2,x_3$ to an output $y$

<div class="note"><b>How to study this notebook:</b> Read once without memorising. Then rerun the code, solve each checkpoint without looking, and finish with the cheat sheet.</div>


## 1. Probability building blocks

| Notation | Meaning |
|---|---|
| $P(A)$ | marginal probability of A |
| $P(A\cap B)$ | joint probability that A and B occur |
| $P(A\mid B)$ | conditional probability of A after learning B |
| $A^c$ | A does not occur |

### Independent events

$A$ and $B$ are independent when knowing one does not change the probability of the other:

$$P(A\mid B)=P(A)$$

Then:

$$P(A\cap B)=P(A)P(B)$$

Repeated fair-die rolls and fair-coin tosses are independent if the physical process does not connect the trials.

### Dependent events

Without replacement, the first draw changes what remains. Always-safe product rule:

$$P(A\cap B)=P(A)P(B\mid A)$$


## 2. Lecture marble example

A bag contains 2 red and 3 yellow marbles. Draw without replacement.

Probability of red first:

$$P(R)=\frac25$$

After red is removed, 3 yellow remain among 4 marbles:

$$P(Y\mid R)=\frac34$$

Joint probability:

$$P(R\cap Y)=\frac25\times\frac34=\frac{6}{20}=0.30$$

Order can change the conditional pieces, but the same joint event can be factorised either way:

$$P(A\cap B)=P(A)P(B\mid A)=P(B)P(A\mid B)$$


## 3. Deriving Bayes’ theorem

Start with the two product-rule forms:

$$P(A\cap B)=P(A)P(B\mid A)$$

$$P(A\cap B)=P(B)P(A\mid B)$$

Set them equal and divide by $P(B)$:

$$\boxed{P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}}$$

Names:

| Component | Name | Question |
|---|---|---|
| $P(A)$ | prior | what did we believe before B? |
| $P(B\mid A)$ | likelihood | how expected is evidence B if A is true? |
| $P(B)$ | evidence / marginal likelihood | how common is B overall? |
| $P(A\mid B)$ | posterior | what should we believe about A after B? |

Memory line:

$$\text{posterior}=\frac{\text{likelihood}\times\text{prior}}{\text{evidence}}$$


## 4. Total probability supplies the denominator

If $A_1,\ldots,A_k$ are mutually exclusive and exhaustive possibilities:

$$P(B)=\sum_{i=1}^{k}P(B\mid A_i)P(A_i)$$

For a binary condition $D$ / not $D$:

$$P(+)=P(+\mid D)P(D)+P(+\mid D^c)P(D^c)$$

This denominator is why **base rates** matter. A strong-looking positive test can still have a modest posterior when the condition is rare.


In [1]:
# Beginner-friendly guide:
# We load tools for calculations, probability distributions, tables, and pictures.
# The next examples turn Bayes updates into numbers that are easier to inspect.
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


In [2]:
# Beginner-friendly guide:
# We combine the starting disease rate with test sensitivity and specificity.
# This shows why a positive result can still have a modest disease chance when the disease is rare.
# Diagnostic-test example
prevalence = 0.01                # P(D)
sensitivity = 0.95               # P(+ | D)
specificity = 0.90               # P(- | not D)
false_positive_rate = 1 - specificity

p_positive = sensitivity*prevalence + false_positive_rate*(1-prevalence)
p_disease_given_positive = sensitivity*prevalence / p_positive

print(f"P(positive) = {p_positive:.4f}")
print(f"P(disease | positive) = {p_disease_given_positive:.4f}")


P(positive) = 0.1085
P(disease | positive) = 0.0876


### Natural-frequency interpretation

Imagine 10,000 people:

- 100 have the condition; about 95 test positive.
- 9,900 do not; with 10% false positives, about 990 test positive.
- Total positives: about 1,085.
- Truly affected among positives: 95.

$$P(D\mid+)\approx\frac{95}{1085}=8.76\%$$

The test is not “95% certain after a positive.” Sensitivity is $P(+\mid D)$; the desired posterior is $P(D\mid+)$. Reversing these is the prosecutor’s fallacy.


## 5. Odds form

Bayes can be written as:

$$\text{posterior odds}=\text{prior odds}\times\text{likelihood ratio}$$

For a positive diagnostic result:

$$LR^+=\frac{\text{sensitivity}}{1-\text{specificity}}$$

Odds are $p/(1-p)$. Convert odds $o$ back to probability using $p=o/(1+o)$.

The odds form makes sequential updating easy: multiply by a new likelihood ratio for each conditionally appropriate piece of evidence.


## 6. Connection to machine learning: Naive Bayes

The lecture maps house features $x_1$ = size, $x_2$ = rooms, $x_3$ = location to an output $y$.

General Bayes classifier:

$$P(y\mid x_1,x_2,x_3)=\frac{P(y)P(x_1,x_2,x_3\mid y)}{P(x_1,x_2,x_3)}$$

For choosing the most likely class, the denominator is common across classes:

$$\hat y=\arg\max_y P(y)P(x_1,x_2,x_3\mid y)$$

Naive Bayes adds a strong assumption: features are conditionally independent given the class.

$$P(x_1,x_2,x_3\mid y)\approx\prod_j P(x_j\mid y)$$

This often works surprisingly well for text classification. For a continuous price target, ordinary Naive Bayes is not the usual model; regression or a Bayesian regression model is more natural.


## 7. Bayesian parameter estimation: Beta–Binomial

Suppose the unknown coin-head probability is $\theta$.

- Prior: $\theta\sim\text{Beta}(a,b)$.
- Data: $h$ heads and $t$ tails.
- Posterior: $\theta\mid data\sim\text{Beta}(a+h,b+t)$.

The prior behaves like pseudo-counts. A uniform Beta(1,1) prior plus 7 heads and 3 tails produces Beta(8,4).

Posterior mean:

$$E[\theta\mid data]=\frac{a+h}{a+b+h+t}$$

Unlike a frequentist p-value, a Bayesian posterior can make probability statements about a parameter—conditional on the prior and likelihood model.


In [3]:
# Beginner-friendly guide:
# We start with a neutral Beta belief, add 7 heads and 3 tails, then describe the updated belief.
# The credible interval is a Bayesian range for the unknown chance of heads.
a, b = 1, 1
heads, tails = 7, 3
post_a, post_b = a + heads, b + tails
posterior_mean = post_a / (post_a + post_b)
credible = stats.beta.ppf([0.025, 0.975], post_a, post_b)

print(f"Posterior: Beta({post_a}, {post_b})")
print(f"Posterior mean: {posterior_mean:.3f}")
print(f"95% credible interval: ({credible[0]:.3f}, {credible[1]:.3f})")


Posterior: Beta(8, 4)
Posterior mean: 0.667
95% credible interval: (0.390, 0.891)


## 8. Frequentist CI versus Bayesian credible interval

| Interval | Interpretation |
|---|---|
| 95% confidence interval | 95% of intervals from repeated use of the procedure cover the fixed parameter |
| 95% credible interval | given the prior, likelihood, and observed data, the parameter has 95% posterior probability inside the interval |

Neither is automatically “better”. They answer different questions and rely on different assumptions.


# End-of-topic cheat sheet

| Concept | Formula |
|---|---|
| Conditional probability | $P(A\mid B)=P(A\cap B)/P(B)$ |
| Product rule | $P(A\cap B)=P(A)P(B\mid A)$ |
| Independence | $P(A\mid B)=P(A)$ |
| Bayes | $P(A\mid B)=P(B\mid A)P(A)/P(B)$ |
| Total probability | $P(B)=\sum_iP(B\mid A_i)P(A_i)$ |
| Posterior odds | prior odds × likelihood ratio |
| Naive Bayes | $P(y\mid x)\propto P(y)\prod_jP(x_j\mid y)$ |
| Beta update | Beta$(a,b)$ → Beta$(a+h,b+t)$ |

**Do not reverse:** $P(A\mid B)$ is generally not equal to $P(B\mid A)$.


# Revision questions and answers

**Q1. What is conditional probability?**

<details><summary>Answer</summary>

The probability of A after learning that B occurred: $P(A\mid B)$.

</details>

---

**Q2. When can P(A and B) be written as P(A)P(B)?**

<details><summary>Answer</summary>

When A and B are independent.

</details>

---

**Q3. State Bayes’ theorem.**

<details><summary>Answer</summary>

$P(A\mid B)=P(B\mid A)P(A)/P(B)$.

</details>

---

**Q4. What is the prior?**

<details><summary>Answer</summary>

The probability distribution or belief about the unknown quantity before the current evidence is used.

</details>

---

**Q5. Why does prevalence matter in diagnostic testing?**

<details><summary>Answer</summary>

It controls the prior odds; with a rare condition, false positives can outnumber true positives.

</details>

---

**Q6. What is the prosecutor’s fallacy?**

<details><summary>Answer</summary>

Confusing a probability of evidence given innocence/guilt with the probability of innocence/guilt given the evidence.

</details>

---

**Q7. What makes Naive Bayes ‘naive’?**

<details><summary>Answer</summary>

It assumes features are conditionally independent given the class.

</details>

---

**Q8. What is the posterior after Beta(1,1), 7 heads and 3 tails?**

<details><summary>Answer</summary>

Beta(8,4).

</details>


## Friendly theory: Bayes’ theorem updates a belief after new evidence arrives

> **This is an extra, simple-language companion. The detailed notes above stay exactly as they are.**

Imagine you are a detective. Before a clue arrives, you have a starting belief called the **prior**. After you see the clue, Bayes’ theorem helps you make an updated belief called the **posterior**.

### A tiny health-test story

Even a very accurate test can produce many false positives when the disease is rare. Why? Because there may be a huge number of healthy people and only a few sick people. The starting rate—the **base rate**—matters a lot.

### Bayes in plain words

**Updated chance = how well the evidence fits the idea × starting chance ÷ chance of seeing that evidence at all.**

### Remember it

Do not flip conditional probabilities around. “Chance of a positive test if sick” is not the same as “chance of being sick after a positive test.”
